# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [2]:
from openai import OpenAI
import pymupdf
import os
from unstructured.chunking.basic import chunk_elements
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel
client = OpenAI()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100, length_function=len)
class Output(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    input_tokens: int
    output_tokens: int

In [3]:
file_path = "./02_activities/data/"
file_name = "ai_report_2025.pdf"

In [4]:
def process_pdf(file_name, file_path = './data/'):
    file_path = os.path.join(file_path, file_name)
    pages = pymupdf.open(file_path)

    documents = ""
    for page in pages:
        documents += page.get_text()
    return documents

text = process_pdf(file_name)
print(text[:500])

pg. 1 
 
 
The GenAI Divide 
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI in


In [29]:
prompt = f"""
    You are a caveman from the Stone Age. 
    Given the following context from a article, do the following:
    
    1. Identify the title and author.
    2. Explain the relevance of this article to AI professionals in a paragraph.
    3. Summarize concisely in no more than 1000 tokens".
        
    The book is the following: 
    <book>
    {text}
    </book>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
"""
response = client.responses.parse(
    model="gpt-4o",
    input = prompt,
    text_format=Output
)

print(response.output_parsed)

author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' title='The GenAI Divide: State of AI in Business 2025' relevance="This article is crucial for AI professionals as it details the challenges and success factors in implementing generative AI (GenAI) in businesses. It highlights the 'GenAI Divide,' where despite substantial investment, only a small fraction of AI projects yield significant returns. Understanding this divide helps AI professionals align their strategies with successful patterns, focusing on adaptability, workflow integration, and learning capabilities of AI systems. The insights can guide professionals to maximize the impact of AI initiatives, ensuring they cross the divide from pilot projects to full deployment, which is essential for driving meaningful business transformation." summary="The 'GenAI Divide' describes the gap between high adoption and low transformation in AI implementation in businesses by 2025. Out of significant enterprise investme

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [30]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
from deepeval.metrics import SummarizationMetric

In [33]:
test_case = LLMTestCase(input=text, actual_output=response.output_parsed.summary)
metric = SummarizationMetric(
    threshold=0.5,
    model="gpt-4o",
    assessment_questions=[
        "Are a majority of companies implementing LLMs into their workflows?",
        "Do the vast majority of AI projects fail to reach production?",
        "Are the majority of cost-savings from AI assistance realized in the front office?"
        "Are companies implementing generative AI seeing a measurable cost reduction?"
        "Are companies implementing generative AI seeing a decrease in staff count?"
    ]
)

In [34]:
evaluate(test_cases=[test_case], metrics=[metric])

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o, strict=False, async_mode=True)...

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************cV8A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
